In [0]:
import requests
products_url = "https://dummyjson.com/products"
Users_url = "https://dummyjson.com/users"
carts_url = "https://dummyjson.com/carts"
posts_url = "https://dummyjson.com/posts"
comments_url = "https://dummyjson.com/comments"
todos_url = "https://dummyjson.com/todos"
products_dic = requests.get(products_url).json()
users_dic = requests.get(Users_url).json()
carts_dic = requests.get(carts_url).json()
posts_dic = requests.get(posts_url).json()
comments_dic = requests.get(comments_url).json()
todos_dic = requests.get(todos_url).json()

In [0]:
import pandas as pd
products_df = pd.DataFrame(products_dic["products"])
users_df = pd.DataFrame(users_dic["users"])
carts_df = pd.DataFrame(carts_dic["carts"])
posts_df = pd.DataFrame(posts_dic["posts"])
comments_df = pd.DataFrame(comments_dic["comments"])
todos_df = pd.DataFrame(todos_dic["todos"])

In [0]:
df_names = {
    "products_df": products_df,
    "users_df": users_df,
    "carts_df": carts_df,
    "posts_df": posts_df,
    "comments_df": comments_df,
    "todos_df": todos_df
}

In [0]:
df_names["comments_df"] = df_names["comments_df"].rename(
    columns={"id": "comment_id"}
)

In [0]:
primary_keys = {
    "products_df": "id",
    "users_df": "id",
    "carts_df": "id",
    "posts_df": "id",
    "comments_df": "comment_id",
    "todos_df": "id"
}


In [0]:
 ## fix dictionary/list types in the main DataFrames

def Clean_df_flatten_dict(df_names):

    normalized_dfs = {}
## copy df to work on it instead of working on the main df
    for name, df in df_names.items():
        df1 = df.copy()
## flatten dictionary cols
        for col in df1.columns.copy():
            if df1[col].apply(lambda x: isinstance(x, dict)).any():
                dict_dfs = pd.json_normalize(df1[col])
                if name == "comments_df" and col == "user":
                    dict_dfs = dict_dfs.rename(columns={"id": "user_id"})
                df1 = pd.concat([df1.drop(columns=[col]), dict_dfs], axis=1)
                
        normalized_dfs[name] = df1
    return normalized_dfs

In [0]:
normalized_dfs = Clean_df_flatten_dict(df_names)

In [0]:
def Clean_df_explode_list(normalized_dfs):
    list_dfs = {}
## explode list cols
    for name, df in normalized_dfs.items():
        df2 = df.copy()
        for col in df2.columns.copy():
            list_dfs[name] = df2
            if df2[col].apply(lambda x: isinstance(x, list)).any():
                pk = primary_keys[name]
                exploded_df = df2[[pk, col]].explode(col).reset_index(drop=True)

                exploded_df = exploded_df.rename(
                    columns={pk: f"{name.replace('_df', '')}_id"}
                )
                list_dfs[f"{name.replace('_df', '')}_{col}_df"] = exploded_df
    return list_dfs

In [0]:
list_dfs = Clean_df_explode_list(normalized_dfs)

In [0]:
## flatten dictionary inside list columns
def Clean_df_flatten_child_dicts(list_dfs):
    all_dfs = {}
    for name, df3 in list_dfs.items():
        for col in df3.columns.copy():
            if df3[col].apply(lambda x: isinstance(x, dict)).any():
                dict_dfs = pd.json_normalize(df3[col])
                df3 = pd.concat([df3.drop(columns=[col]), dict_dfs], axis=1)
                
        all_dfs[name] = df3
    return all_dfs

In [0]:
all_dfs = Clean_df_flatten_child_dicts(list_dfs)

In [0]:
for name, df in normalized_dfs.items():
    print(f"\n--- {name} ---")
    for col in df.columns:
        types = df[col].apply(type).value_counts()
        print(f"{col}: {types.to_dict()}")


--- products_df ---
id: {<class 'int'>: 30}
title: {<class 'str'>: 30}
description: {<class 'str'>: 30}
category: {<class 'str'>: 30}
price: {<class 'float'>: 30}
discountPercentage: {<class 'float'>: 30}
rating: {<class 'float'>: 30}
stock: {<class 'int'>: 30}
tags: {<class 'list'>: 30}
brand: {<class 'str'>: 15, <class 'float'>: 15}
sku: {<class 'str'>: 30}
weight: {<class 'int'>: 30}
warrantyInformation: {<class 'str'>: 30}
shippingInformation: {<class 'str'>: 30}
availabilityStatus: {<class 'str'>: 30}
reviews: {<class 'list'>: 30}
returnPolicy: {<class 'str'>: 30}
minimumOrderQuantity: {<class 'int'>: 30}
images: {<class 'list'>: 30}
thumbnail: {<class 'str'>: 30}
width: {<class 'float'>: 30}
height: {<class 'float'>: 30}
depth: {<class 'float'>: 30}
createdAt: {<class 'str'>: 30}
updatedAt: {<class 'str'>: 30}
barcode: {<class 'str'>: 30}
qrCode: {<class 'str'>: 30}

--- users_df ---
id: {<class 'int'>: 30}
firstName: {<class 'str'>: 30}
lastName: {<class 'str'>: 30}
maidenName:

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.API_TABLES;

In [0]:
all_dfs.keys()

dict_keys(['products_df', 'products_tags_df', 'products_reviews_df', 'products_images_df', 'users_df', 'carts_df', 'carts_products_df', 'posts_df', 'posts_tags_df', 'comments_df', 'todos_df'])

In [0]:
## Loading inside databricks tables

for name,df in all_dfs.items():
    table_name = name.replace("_df", "")
    spark_df = spark.createDataFrame(df)

    spark_df.write.format("delta").saveAsTable(
        f"workspace.API_TABLES.{table_name}"
    )

In [0]:
for name, df in all_dfs.items():
    table = name.replace("_df", "")

    spark_count = spark.table(f"workspace.API_TABLES.{table}").count()
    pandas_count = len(df)

    print(f"{table}: Spark = {spark_count}, Pandas = {pandas_count}")

products: Spark = 30, Pandas = 30
products_tags: Spark = 47, Pandas = 47
products_reviews: Spark = 90, Pandas = 90
products_images: Spark = 54, Pandas = 54
users: Spark = 30, Pandas = 30
carts: Spark = 30, Pandas = 30
carts_products: Spark = 114, Pandas = 114
posts: Spark = 30, Pandas = 30
posts_tags: Spark = 82, Pandas = 82
comments: Spark = 30, Pandas = 30
todos: Spark = 30, Pandas = 30
